# VeloceReduction — Reduce one observing night

This notebook is the master workflow for reducing a complete Veloce observing
night.

**Input:** `observations/YYMMDD/`  
**Output:** `reductions/vr_X.Y.Z/YYMMDD/`

Detailed algorithms are implemented in the `velocereduction` Python modules.
This notebook intentionally contains only the main reduction steps.

## 0. Setup

Choose the night and reduction settings.

When run interactively as a notebook, the values below are used directly.
When converted to `reduce_night.py`, the night is supplied on the command line.

In [ ]:
import sys
import argparse

import numpy as np

from astropy.table import Table

import matplotlib.pyplot as plt

from velocereduction import (
    __version__,
    utils,
    flat,
    tramlines,
    calibration,
    wavelength
)


def running_in_notebook():
    """True for an interactive Jupyter notebook, False for reduce_night.py."""
    if 'ipykernel' in sys.modules:
        return True
    else:
        return False


IN_NOTEBOOK = running_in_notebook()

In [ ]:
if IN_NOTEBOOK:

    # -------------------------------------------------------------------------
    # Interactive notebook settings
    # -------------------------------------------------------------------------

    night = '001122'            # Observing night in YYMMDD format,
                                # e.g. 001122, which is the reference night
    # night = '260703'            

    log_level = 'DEBUG'          # DEBUG / INFO / WARNING / ERROR
    diagnostics = 'full'       # none / basic / full

    extraction_mode = 'summed'  # summed / fibre
    overwrite = False

else:

    # -------------------------------------------------------------------------
    # Command-line settings for reduce_night.py
    # -------------------------------------------------------------------------

    parser = argparse.ArgumentParser(
        description='Reduce one complete Veloce observing night.'
    )

    parser.add_argument(
        'night',
        help='Observing night in YYMMDD format, e.g. 001122'
    )

    parser.add_argument(
        '--log-level',
        choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'],
        default='INFO'
    )

    parser.add_argument(
        '--diagnostics',
        choices=['none', 'basic', 'full'],
        default='basic'
    )

    parser.add_argument(
        '--extraction-mode',
        choices=['summed', 'fibre'],
        default='summed'
    )

    parser.add_argument(
        '--overwrite',
        action='store_true'
    )

    args = parser.parse_args()

    night = args.night
    log_level = args.log_level
    diagnostics = args.diagnostics
    extraction_mode = args.extraction_mode
    overwrite = args.overwrite

In [ ]:
config = utils.ReductionConfig(
    night=night,
    log_level=log_level,
    diagnostics=diagnostics,
    extraction_mode=extraction_mode,
    overwrite=overwrite,
)

paths = utils.prepare_reduction(config, version=__version__)
logger = utils.setup_logging(config, paths)

logger.info('Starting VeloceReduction %s for night %s', __version__, night)
logger.info('Diagnostics: %s', diagnostics)
logger.info('Extraction mode: %s', extraction_mode)

print(f'\nNight:             {night}')
print(f'VeloceReduction:   {__version__}')
print(f'Logging:           {log_level}')
print(f'Diagnostics:       {diagnostics}')
print(f'Extraction:        {extraction_mode}')
print(f'Output:            {paths.root}')

## 1. Identify observations

**Input:** raw observations and observing log  
**Output:** night overview and `reduction_input_YYMMDD.txt`

All observations are classified once at the beginning of the reduction.
Later stages use this table rather than repeatedly reading and classifying
raw FITS headers.

In [ ]:
reduction_input = utils.identify_observations(config, paths)

utils.write_reduction_input(reduction_input, config, paths)

if IN_NOTEBOOK:
    display(reduction_input)

## 2. Measure displacement relative to reference night

Measure the displacement of each detector relative to the reference night.

SimTh images are used for all three CCDs, with SimLC providing an additional
registration measurement for CCD2 and CCD3.

In [ ]:
detector_shifts = tramlines.measure_detector_shifts(
    reduction_input,
    config,
    paths,
)

if IN_NOTEBOOK:
    display(detector_shifts)

## 3. Master Flat, nightly tramlines, and Flat response

**Input:** Flat observations + reference tramlines + detector shifts  
**Output:** master Flat, nightly tramline geometry, extracted Flat, response Flat, and blaze function

Create a high-S/N master Flat in detector coordinates and use it to determine
the nightly tramline geometry. The Flat is then extracted along the fitted
tramlines and smoothed along the fibre direction to separate the small-scale
detector response from the smooth illumination and blaze.

In [ ]:
master_flat = flat.create_master_flat(
    reduction_input,
    config,
    paths,
)

In [ ]:
nightly_tramlines = tramlines.fit_nightly_tramlines(
    reduction_input,
    master_flat,
    detector_shifts,
    config,
    paths,
)

if (
    IN_NOTEBOOK
    and config.diagnostics != 'none'
):
    tramlines.show_summary(
        nightly_tramlines
    )

In [ ]:
flat_products = flat.create_flat_products(
    master_flat,
    nightly_tramlines,
    config,
    paths,
)

## 6. Extract wavelength-calibration observations

**Input:** SimLC and FibTh observations  
**Output:** time-stamped extracted calibration spectra

SimLC and FibTh are both used to establish the wavelength solution.

In [ ]:
calibration_spectra = tramlines.extract_calibration_spectra(
    reduction_input,
    nightly_tramlines,
    config,
    paths
)

In [ ]:
for calibration_type in ['SimTh','SimLC','FibTh']:
    tramlines.save_calibration_spectra(
        calibration_spectra[calibration_type],
        paths.wavelength / f'{calibration_type.lower()}.fits',
        calibration_type=calibration_type,
        overwrite=True,
    )

In [ ]:
calibration_peak_tables = calibration.measure_calibration_peaks_for_night(
    calibration_spectra,
    output_dir=paths.wavelength,
    diagnostic_dir=paths.figures / 'calibration',
    maximum_signal={
        'SimLC': None,
        'SimTh': None,
        'FibTh': None,
    },
    log_level=config.log_level,
    diagnostics=config.diagnostics,
    overwrite=config.overwrite
)

## 7. Fit the wavelength solution

**Input:** extracted SimLC + FibTh spectra and their MJD timestamps  
**Output:** wavelength calibration model for the night

In [ ]:
initial_wavelength_solution = (
    wavelength.load_initial_wavelength_solutions(
        paths.repository
    )
)

# VELOCE_CCD_ORDERS = {
#     "1": np.arange(167, 138 - 1, -1),
#     "2": np.arange(140, 103 - 1, -1),
#     "3": np.arange(104, 65 - 1, -1),
# }

# for ccd in [1, 2, 3]:
#     for type in ['SimTh','SimLC','FibTh']:
#         try:
#             all_coefficients = Table.read(paths.calibrations / f'wavelength/simlc_ccd{ccd}_wavelength_solution.fits',2)
#         except:
#             all_coefficients = Table.read(paths.calibrations / f'wavelength/fibth_ccd{ccd}_wavelength_solution.fits',2)

#         for order in VELOCE_CCD_ORDERS[str(ccd)]:
#             order_name = f'ccd_{ccd}_order_{order}'

#             coefficients = all_coefficients[all_coefficients['order'] == order]

#             initial_wavelength_solution[type][order_name] = np.array([coefficients['coefficient_'+str(i)][0] for i in range(0, 7)])

In [ ]:
calibration_peak_tables["SimLC"] = (
    wavelength.identify_simlc_peaks(
        calibration_peak_tables["SimLC"],
        initial_wavelength_solution["SimLC"],
        detector_shifts,
    )
)

In [ ]:
simlc_wavelength_solutions = {}

for ccd in [2, 3]:

    simlc_wavelength_solutions[ccd] = (
        wavelength.fit_and_save_wavelength_surface(
            calibration_peak_tables["SimLC"],

            calibration_type="SimLC",
            ccd=ccd,

            degree_y=7,
            degree_m=5,

            minimum_y=200,
            maximum_y=3900,
            minimum_signal_to_noise=20,
            maximum_y_uncertainty=0.01,
            sigma_clip = 3.0,

            output_dir=paths.wavelength,
            diagnostic_dir=(
                paths.figures
                / "wavelength"
            ),

            diagnostics=config.diagnostics,
            overwrite=True,
        )
    )

# Update for FibTh

In [ ]:
# ThAr atlas from Murphy et al. (2007) with update from 090311
dtype = [
    ("wavenumber", float),
    ("wave_air", float),
    ("log10_intensity", float),
    ("element", "U10"),
    ("ion", "U10"),
    ("source", "U2"),
]
thar_lines = Table(np.genfromtxt(
    paths.repository / 'velocereduction/veloce_reference_data/thar_UVES_MM090311.dat',
    dtype=dtype,
    comments="#",
    autostrip=True
))
thar_lines['wave_vac'] = utils.wavelength_air_to_vac(thar_lines['wave_air'])


In [ ]:
# Match fitted ThXe peaks to ThAr atlas lines

VELOCE_CCD_ORDERS = {
    "1": np.arange(167, 138 - 1, -1),
    "2": np.arange(140, 103 - 1, -1),
    "3": np.arange(104, 65 - 1, -1),
}

calibration_peak_tables["FibTh"]['closest_th_wavelength'] = np.zeros(len(calibration_peak_tables["FibTh"]), dtype=float)
calibration_peak_tables["FibTh"]['closest_th_wavelength'][:] = np.nan

for ccd in [1, 2, 3]:
    for order in VELOCE_CCD_ORDERS[str(ccd)]:
        indices = np.arange(len(calibration_peak_tables["FibTh"]))
        relevant_peak_indices = indices[(
            (calibration_peak_tables["FibTh"]["ccd"] == ccd)
            & (calibration_peak_tables["FibTh"]["order"] == order)
            & (calibration_peak_tables["FibTh"]["used_for_wavelength_fit"] == True)
        )]

        # y_values = np.arange(4112) - 2048 + detector_shifts['dy'][detector_shifts['ccd'] == ccd].value
        y_values = np.array(calibration_peak_tables["FibTh"][relevant_peak_indices]['y']) - 2048 + detector_shifts['dy'][detector_shifts['ccd'] == ccd].value

        expected_wavelength = np.array(
            np.polynomial.polynomial.polyval(
                y_values,
                initial_wavelength_solution['FibTh'][f'ccd_{ccd}_order_{order}'],
            )
        ) * 10.

        reference_th_lines_in_range = (
            (thar_lines['wave_vac'] > expected_wavelength.min())
            & (thar_lines['wave_vac'] < expected_wavelength.max())
            & (thar_lines['element'] == 'Th')
        )

        print(f'CCD {ccd}, order {order}: {len(calibration_peak_tables["FibTh"][relevant_peak_indices])} FibTh peaks, {len(thar_lines[reference_th_lines_in_range])} ThAr lines in range')

        # now loop over the FibTh peaks and find the closest ThAr line in the atlas
        for peak_index, original_index in enumerate(relevant_peak_indices):

            closest_th_line = np.argmin(
                np.abs(
                    thar_lines[reference_th_lines_in_range]['wave_vac']
                    - expected_wavelength[peak_index]
                )
            )

            closest_wave = thar_lines[reference_th_lines_in_range]['wave_vac'][closest_th_line]

            if np.abs(closest_wave - expected_wavelength[peak_index]) < 0.5:
                calibration_peak_tables["FibTh"][
                    'closest_th_wavelength'
                ][original_index] = closest_wave

In [ ]:
fibth_wavelength_solutions = {}

for ccd in [1, 2, 3]:

    fibth_wavelength_solutions[ccd] = (
        wavelength.fit_and_save_wavelength_surface(
            calibration_peak_tables["FibTh"],

            calibration_type="FibTh",
            ccd=ccd,

            degree_y=7,
            degree_m=5,

            minimum_y=200,
            maximum_y=3900,
            minimum_signal_to_noise=10,
            maximum_y_uncertainty=0.1,
            sigma_clip = 3.0,

            output_dir=paths.wavelength,
            diagnostic_dir=(
                paths.figures
                / "wavelength"
            ),

            diagnostics=config.diagnostics,
            overwrite=True,
        )
    )

# AND NOW ON TO FibTh and SimTh!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

profile = np.array([
    81.57262, 82.74879, 84.07484, 85.51365, 87.305565,
    89.30765, 91.70762, 95.04747, 103.25724, 187.9351,
    856.6781, 1623.3531, 1191.6061, 1605.9312, 1380.1094,
    382.45584, 176.25848, 604.99426, 1762.9586, 1599.2748,
    1561.6721, 1962.3041, 1625.9062, 2022.3882, 1294.7383,
    1659.822, 1832.4697, 1595.2172, 1885.2881, 1115.4753,
    1632.6721, 1746.5565, 1762.3352, 2053.2246, 1477.6094,
    2021.2198, 1392.0208, 1542.4307, 2038.9489, 1626.499,
    2133.77, 1510.3711, 1855.7716, 1824.5089, 1558.361,
    2057.669, 1420.9939, 1948.443, 1699.8217, 1825.9548,
    2228.4048, 1639.69, 2232.546, 1675.7391, 2038.3729,
    1905.6105, 1585.1754, 2203.74, 1496.3717, 2093.3496,
    1956.5414, 1926.3536, 1942.1548, 606.71, 186.84503,
    487.95825, 1596.2225, 1678.0306, 1461.7289, 1809.1649,
    1252.9032, 1883.7676, 1356.7611, 295.69467, 111.47796,
    96.30301, 91.46026, 88.387535, 86.011986, 83.81893,
    82.0248
])

# Pixel coordinates, with the central profile pixel at x=0
x = np.arange(len(profile)) - (len(profile) - 1) / 2


# ---------------------------------------------------------------------
# Fibre geometry
#
# left sky:       -12, -11
# blank:          -10
# science:         -9 ... +9       (19 fibres)
# blank:          +10
# right sky:      +11, +12, +13
#
# Actual position = x0 + slot * separation
# ---------------------------------------------------------------------

sky_left_slots  = np.array([-12, -11])
science_slots   = np.arange(-9, 10)
sky_right_slots = np.array([11, 12, 13])

fibre_slots = np.concatenate([
    sky_left_slots,
    science_slots,
    sky_right_slots
])

n_fibres = len(fibre_slots)   # 24


def gaussian_model(params, x):
    """
    params =
        [background, x0, separation, sigma, amplitude_0, ..., amplitude_23]
    """

    background = params[0]
    x0         = params[1]
    separation = params[2]
    sigma      = params[3]
    amplitudes = params[4:]

    centres = x0 + separation * fibre_slots

    model = np.full_like(x, background, dtype=float)

    for amplitude, centre in zip(amplitudes, centres):
        model += amplitude * np.exp(
            -0.5 * ((x - centre) / sigma)**2
        )

    return model


def residuals(params, x, y):
    return gaussian_model(params, x) - y


# ---------------------------------------------------------------------
# Initial guesses
# ---------------------------------------------------------------------

background_init = np.median(
    np.concatenate([profile[:8], profile[-8:]])
)

x0_init = 0.0
separation_init = 2.4
sigma_init = 0.85

centres_init = x0_init + separation_init * fibre_slots

# Estimate each amplitude from the closest pixel
amplitudes_init = []

for centre in centres_init:
    idx = np.argmin(np.abs(x - centre))
    amplitudes_init.append(
        max(profile[idx] - background_init, 10)
    )

amplitudes_init = np.array(amplitudes_init)

p0 = np.concatenate([
    [
        background_init,
        x0_init,
        separation_init,
        sigma_init
    ],
    amplitudes_init
])


# ---------------------------------------------------------------------
# Bounds
# ---------------------------------------------------------------------

lower_bounds = np.concatenate([
    [
        0,       # background
        -1.0,    # central-fibre offset
        2.0,     # separation
        0.3      # sigma
    ],
    np.zeros(n_fibres)
])

upper_bounds = np.concatenate([
    [
        500,     # background
        +1.0,    # central-fibre offset
        2.8,     # separation
        2.0      # sigma
    ],
    np.full(n_fibres, 5000)
])


# ---------------------------------------------------------------------
# Fit
# ---------------------------------------------------------------------

result = least_squares(
    residuals,
    p0,
    args=(x, profile),
    bounds=(lower_bounds, upper_bounds),
    max_nfev=20000
)

popt = result.x

background = popt[0]
x0         = popt[1]
separation = popt[2]
sigma      = popt[3]
amplitudes = popt[4:]

centres = x0 + separation * fibre_slots

model = gaussian_model(popt, x)
resid = profile - model

print(f"Background = {background:.3f}")
print(f"x0         = {x0:.4f} pix")
print(f"Separation = {separation:.4f} pix")
print(f"sigma      = {sigma:.4f} pix")
print(f"FWHM       = {2.35482 * sigma:.4f} pix")
print(f"RMS        = {np.sqrt(np.mean(resid**2)):.3f}")

print("\nFibre centres:")
for i, (slot, centre, amplitude) in enumerate(
    zip(fibre_slots, centres, amplitudes)
):
    if slot in science_slots:
        fibre_type = "Science"
    else:
        fibre_type = "Sky"

    print(
        f"{i:2d}  {fibre_type:7s}  "
        f"slot={slot:+3d}  "
        f"x={centre:+7.3f}  "
        f"A={amplitude:8.1f}"
    )


# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------

xx = np.linspace(x.min(), x.max(), 3000)
model_fine = gaussian_model(popt, xx)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(x, profile, "o", label="Profile")
ax.plot(xx, model_fine, "-", label="Gaussian model")

# Plot individual Gaussian components
for amplitude, centre in zip(amplitudes, centres):

    component = (
        background
        + amplitude
        * np.exp(-0.5 * ((xx - centre) / sigma)**2)
    )

    ax.plot(xx, component, alpha=0.35)

# fibre centres
for slot, centre in zip(fibre_slots, centres):

    if slot in science_slots:
        ax.axvline(centre, lw=0.5, alpha=0.3)
    else:
        ax.axvline(centre, lw=0.5, ls="--", alpha=0.5)

# blank fibre locations
blank_slots = [-10, 10]

for slot in blank_slots:
    blank_centre = x0 + slot * separation
    ax.axvline(
        blank_centre,
        ls=":",
        lw=1.5,
        label="Blank fibre" if slot == -10 else None
    )

ax.set_xlabel("Cross-dispersion pixel relative to central fibre")
ax.set_ylabel("Flux")
ax.legend()

plt.tight_layout()

plt.savefig(
    paths.figures / "calibration/fibre_profile_fit.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import sys
sys.exit()

In [ ]:
import pickle

# ThAr atlas from Murphy et al. (2007) with update from 090311
dtype = [
    ("wavenumber", float),
    ("wave_vac", float),
    ("log10_intensity", float),
    ("element", "U10"),
    ("ion", "U10"),
    ("source", "U2"),
]
thar_lines = Table(np.genfromtxt(
    paths.repository / 'velocereduction/veloce_reference_data/thar_UVES_MM090311.dat',
    dtype=dtype,
    comments="#",
    autostrip=True
))

nist_th_file = paths.repository / 'velocereduction/veloce_reference_data/th_linelist_NIST.pickle'
with open(nist_th_file, 'rb') as f:
    atomic_data_dict = pickle.load(f)
nist_th_lines = dict()
nist_th_lines['wave_air']  = atomic_data_dict['linelist']['obs_wl_air(nm)']*10
nist_th_lines['wave_vac']  = utils.wavelength_air_to_vac(atomic_data_dict['linelist']['obs_wl_air(nm)']*10)
nist_th_lines['intensity'] = atomic_data_dict['linelist']['intens']


wave_begin = 3600
wave_end = 9500
wave_delta = 0.01
fine_wavelengths = np.arange(wave_begin, wave_end, wave_delta)

thar_lines_fine_murphy = dict()
thar_lines_fine_murphy['wave_vac'] = fine_wavelengths
thar_lines_fine_murphy['wave_air'] = utils.wavelength_vac_to_air(fine_wavelengths)
thar_lines_fine_murphy['intensity'] = np.zeros(len(fine_wavelengths))
for index in range(len(thar_lines_fine_murphy['wave_vac'])):
    thar_line_in_range = np.where((thar_lines['wave_vac']-0.5*wave_delta < thar_lines_fine_murphy['wave_vac'][index]) & (thar_lines_fine_murphy['wave_vac'][index] < thar_lines['wave_vac']+0.5*wave_delta))[0]
    if len(thar_line_in_range) > 0:
        thar_lines_fine_murphy['intensity'][index] = np.mean(10**thar_lines['log10_intensity'][thar_line_in_range])

th_lines_fine_nist = dict()
th_lines_fine_nist['wave_vac'] = fine_wavelengths
th_lines_fine_nist['wave_air'] = utils.wavelength_vac_to_air(fine_wavelengths)
th_lines_fine_nist['intensity'] = np.zeros(len(fine_wavelengths))
for index in range(len(th_lines_fine_nist['wave_vac'])):
    th_line_in_range = np.where((nist_th_lines['wave_vac']-0.5*wave_delta < th_lines_fine_nist['wave_vac'][index]) & (th_lines_fine_nist['wave_vac'][index] < nist_th_lines['wave_vac']+0.5*wave_delta))[0]
    if len(th_line_in_range) > 0:
        th_lines_fine_nist['intensity'][index] = np.mean(nist_th_lines['intensity'][th_line_in_range])

In [ ]:
plt.figure(figsize=(15,3))
# plt.plot(
#     th_lines_fine_nist['wave_vac'],
#     th_lines_fine_nist['intensity']
# )
# plt.plot(
#     thar_lines_fine_murphy['wave_vac'],
#     thar_lines_fine_murphy['intensity']
# )
plt.plot(
    thar_lines_fine_murphy['wave_vac'],
    thar_lines_fine_murphy['intensity'] - th_lines_fine_nist['intensity']
)

In [ ]:
thar_lines

In [ ]:
def plot_th(order, ccd, y0=2048):
    # def calc_(y0):

    try:
        simlc_coefficients = Table.read(paths.calibrations / f'wavelength/simlc_ccd{ccd}_wavelength_solution.fits',2)
        simlc_coefficients = simlc_coefficients[simlc_coefficients['order'] == order]
        coefficients = np.array([simlc_coefficients['coefficient_'+str(i)][0] for i in range(0, 7)])
        print('Using 2-dim. SimLC solution')
    except:
        try:
            coefficients = np.loadtxt(paths.repository / f'velocereduction/wavelength_coefficients/wavelength_coefficients_ccd_{ccd}_order_{order}_lc.txt')
        except:
            coefficients = np.loadtxt(paths.repository / f'velocereduction/wavelength_coefficients/wavelength_coefficients_ccd_{ccd}_order_{order}_thxe.txt')

            
    simth_counts = calibration_spectra['SimTh'][str(ccd)][0]['counts'][104-order]

    y = np.arange(4112)
    # y0 = np.median(y)   # 2055.5

    detector_shift_y = detector_shifts['dy'][
        detector_shifts['ccd'] == int(ccd)
    ][0]

    y_centered = y - y0

    wavelength_reference = np.polynomial.polynomial.polyval(
        y_centered,
        coefficients,
    )*10

    wavelength_min = wavelength_reference.min()
    wavelength_max = wavelength_reference.max()

    thar_lines_in_range = (
        (thar_lines['wave_vac'] > wavelength_min)
        & (thar_lines['wave_vac'] < wavelength_max)
    )

    fig, ax = plt.subplots(figsize=(15,3))

    ax.plot(
        wavelength_reference,
        # np.log10(simth_counts),
        simth_counts.clip(max=500),
        lw=0.8,
    )

    print(len(thar_lines[thar_lines_in_range]), 'ThAr lines in range')

    for thar_line in thar_lines[thar_lines_in_range]:
        ax.axvline(
            thar_line['wave_vac'],
            lw=thar_line['log10_intensity']/10,
            color = 'C3'
        )

    # axes[0].plot(
    #     thar_lines_fine_murphy['wave_vac'][th_lines_fine_murphy_in_range],
    #     thar_lines_fine_murphy['intensity'][th_lines_fine_murphy_in_range],#, / np.max(thar_lines_fine_murphy['intensity'][th_lines_fine_murphy_in_range]),
    #     color='C1',
    #     lw=0.5,
    # )
    # axes[0].plot(
    #     th_lines_fine_nist['wave_vac'][th_lines_fine_nist_in_range],
    #     th_lines_fine_nist['intensity'][th_lines_fine_nist_in_range],# / np.max(th_lines_fine_nist['intensity'][th_lines_fine_nist_in_range]),
    #     color='C3',
    #     lw=0.5,
    # )

    # for th_lines_fine_nist_vacuum_i, th_lines_fine_nist_air_i, th_lines_fine_nist_intensity_i in zip(
    #     th_lines_fine_nist['wave_vac'][th_lines_fine_nist_in_range],
    #     th_lines_fine_nist['wave_air'][th_lines_fine_nist_in_range],
    #     th_lines_fine_nist['intensity'][th_lines_fine_nist_in_range],
    # ):
        # axes[0].text(
        #     th_lines_fine_nist_vacuum_i,
        #     th_lines_fine_nist_intensity_i,
        #     f'{th_lines_fine_nist_air_i:.4f}',
        #     rotation=90,
        #     fontsize=6,
        #     ha='center',
        #     va='bottom',
        #     c = 'C0'
        # )
        # axes[0].scatter(
        #     th_lines_fine_nist_vacuum_i,
        #     th_lines_fine_nist_intensity_i,
        #     s=10,
        #     c='C0',
        #     alpha=0.5,
        # )
        # axes[0].axvline(
        #     th_lines_fine_nist_vacuum_i,
        #     lw=0.7,
        #     alpha=0.5,
        #     c = 'C0'
        # )

    # for murphy_wavelength_vacuum_i, murphy_wavelength_air_i, murphy_intensity_i in zip(
    #     thar_lines_fine_murphy['wave_vac'][th_lines_fine_murphy_in_range],
    #     thar_lines_fine_murphy['wave_air'][th_lines_fine_murphy_in_range],
    #     thar_lines_fine_murphy['intensity'][th_lines_fine_murphy_in_range],
    # ):
        # axes[0].text(
        #     murphy_wavelength_vacuum_i,
        #     murphy_intensity_i,
        #     f'{murphy_wavelength_air_i:.4f}',
        #     rotation=90,
        #     fontsize=6,
        #     ha='center',
        #     va='bottom',
        #     c = 'C1'
        # )
        # axes[0].scatter(
        #     murphy_wavelength_vacuum_i,
        #     murphy_intensity_i,
        #     s=10,
        #     c='C1',
        #     alpha=0.5,
        # )
        # axes[0].axvline(
        #     murphy_wavelength_vacuum_i,
        #     lw=0.7,
        #     alpha=0.5,
        #     c = 'C1'
        # )


    # axes[0].set_ylabel('Wavelength')
    # axes[0].set_title(
    #     'Murphy Th wavelengths: AIR'
    # )


    # axes[1].plot(
    #     wavelength_reference,
    #     np.log10(simth_counts.clip(min=1)),
    #     lw=0.8,
    # )

    # axes[1].set_ylabel('Counts')
    # axes[1].set_xlabel(
    #     'Dispersion pixel $y$'
    # )

    # axes[1].set_title(
    #     'Murphy Th wavelengths: VACUUM'
    # )

    plt.show()
    plt.close()

plot_th(order=87, ccd=3, y0=2048)

## 8. Dark Products

In [ ]:
# dark_products = utils.process_darks(
#     processed,
#     config,
#     paths
# ) 

## 9. Extract science spectra

**Input:** processed Science images + tramlines + Flat products + wavelength model  
**Output:** wavelength-calibrated Science spectra

The current default extraction is summed across the Science fibres.
A fibre-resolved extraction can use the same interface in future.

In [ ]:
# science_spectra = extraction.extract_science(
#     processed.science,
#     nightly_tramlines,
#     flat_products,
#     wavelength_solution,
#     config,
#     paths
# )

## 10. Velocities

**Input:** wavelength-calibrated Science spectra  
**Output:** barycentric corrections and first-pass radial velocities

In [ ]:
# science_spectra = velocities.add_barycentric_corrections(
#     science_spectra,
#     config,
#     paths
# )

# science_spectra = velocities.measure_initial_rvs(
#     science_spectra,
#     template='solar',
#     config=config,
#     paths=paths
# )

## 11. B-star and telluric products

**Input:** B-star observations  
**Output:** extracted B-star spectra and telluric measurements

In [ ]:
# bstar_spectra = extraction.extract_science(
#     processed.bstars,
#     nightly_tramlines,
#     flat_products,
#     wavelength_solution,
#     config,
#     paths,
#     product_type='Bstar'
# )

# telluric_products = tellurics.measure_tellurics(
#     bstar_spectra,
#     config,
#     paths
# )

## 12. Final products and reduction summary

Write the final Science FITS products, retained diagnostic figures, and
human-readable summary of the night.

In [ ]:
# utils.write_science_products(
#     science_spectra,
#     config,
#     paths
# )

# summary = utils.create_reduction_summary(
#     reduction_input=reduction_input,
#     detector_shifts=detector_shifts,
#     tramlines=nightly_tramlines,
#     wavelength_solution=wavelength_solution,
#     science_spectra=science_spectra,
#     config=config,
#     paths=paths
# )

# utils.write_reduction_summary(summary, config, paths)

In [ ]:
logger.info('Reduction completed successfully.')

print()
print('Reduction complete')
print(f'Night:    {night}')
print(f'Version:  {__version__}')
print(f'Products: {paths.root}')
print(f'Summary:  {paths.reduction_summary}')